# Energy Demand Forecasting using ARIMA and SARIMA

This notebook builds **ARIMA** and **SARIMA** models for hourly energy demand forecasting.

## Goals
- Clean the dataset in a consistent way
- Use the **same target column** for both models
- Use a **chronological split** so there is no data leakage
- Compare model performance using **MAE**, **RMSE**, and **MAPE**

> Expected file: `energy_dataset.csv`


In [ ]:
# Uncomment this only if the libraries are not installed yet
# !pip install pandas numpy matplotlib scikit-learn statsmodels


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX


## 1. Load the dataset

Make sure `energy_dataset.csv` is in the same folder as this notebook.


In [ ]:
# Load the dataset
df = pd.read_csv("energy_dataset.csv")

# Show the first rows
df.head()


## 2. Basic cleaning

This section:
- parses the time column
- sorts the data
- keeps only the target column
- handles missing values
- resamples to hourly frequency to make the time steps consistent


In [ ]:
# Parse the time column safely
df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")

# Remove rows with invalid timestamps
df = df.dropna(subset=["time"]).copy()

# Sort by time so the sequence is correct
df = df.sort_values("time")

# Set time as index
df = df.set_index("time")

# Select the target column only for ARIMA / SARIMA
target_col = "total load actual"
ts = df[[target_col]].copy()

# Remove duplicate timestamps, keeping the first occurrence
ts = ts[~ts.index.duplicated(keep="first")]

# Resample to hourly data in case there are missing or uneven intervals
ts = ts.resample("H").mean()

# Fill missing values using time interpolation, then forward/back fill as backup
ts[target_col] = ts[target_col].interpolate(method="time")
ts[target_col] = ts[target_col].ffill().bfill()

# Remove obvious outliers with the IQR rule, then refill the small gaps created
q1 = ts[target_col].quantile(0.25)
q3 = ts[target_col].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

ts.loc[(ts[target_col] < lower) | (ts[target_col] > upper), target_col] = np.nan
ts[target_col] = ts[target_col].interpolate(method="time").ffill().bfill()

print("Cleaned shape:", ts.shape)
ts.head()


## 3. Visualize the cleaned demand series


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(ts.index, ts[target_col])
plt.title("Cleaned Hourly Energy Demand")
plt.xlabel("Time")
plt.ylabel(target_col)
plt.show()


## 4. Split the data chronologically

We use the **last 20%** as test data.
This is important for time-series forecasting because we must never shuffle the data.


In [ ]:
split_idx = int(len(ts) * 0.80)

train = ts.iloc[:split_idx].copy()
test = ts.iloc[split_idx:].copy()

print("Train size:", len(train))
print("Test size:", len(test))
print("Train range:", train.index.min(), "to", train.index.max())
print("Test range:", test.index.min(), "to", test.index.max())


## 5. Stationarity check with ADF test

ARIMA works better when the series is stationary.
If the p-value is greater than 0.05, differencing is usually needed.


In [ ]:
def adf_test(series, name="Series"):
    result = adfuller(series.dropna())
    print(f"{name} ADF Statistic: {result[0]:.4f}")
    print(f"{name} p-value      : {result[1]:.6f}")
    print("Critical Values:")
    for key, value in result[4].items():
        print(f"  {key}: {value:.4f}")

adf_test(train[target_col], name="Training demand")


## 6. Fit ARIMA

You can tune the `(p, d, q)` values later if needed.
The values below are a reasonable baseline.


In [ ]:
# Baseline ARIMA order
arima_order = (2, 1, 2)

# Fit the ARIMA model on the training data only
arima_model = ARIMA(train[target_col], order=arima_order)
arima_result = arima_model.fit()

# Forecast the length of the test set
arima_forecast = arima_result.forecast(steps=len(test))

# Put the forecast into a Series with the test index
arima_forecast = pd.Series(arima_forecast, index=test.index, name="ARIMA Forecast")

arima_result.summary()


## 7. Fit SARIMA

SARIMA is useful when the data has seasonality.
For hourly energy demand, a daily cycle often appears, so a seasonal period of **24** is a good starting point.


In [ ]:
# Baseline SARIMA settings
sarima_order = (1, 1, 1)
seasonal_order = (1, 1, 1, 24)

# Fit the SARIMA model on the training data only
sarima_model = SARIMAX(
    train[target_col],
    order=sarima_order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_result = sarima_model.fit(disp=False)

# Forecast the test horizon
sarima_forecast = sarima_result.forecast(steps=len(test))
sarima_forecast = pd.Series(sarima_forecast, index=test.index, name="SARIMA Forecast")

sarima_result.summary()


## 8. Evaluate the models

We use:
- **MAE**: average absolute error
- **RMSE**: penalizes larger errors more
- **MAPE**: percentage error


In [ ]:
def evaluate_forecast(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)

    # Avoid division by zero in MAPE
    non_zero_mask = y_true != 0
    mape = (np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask]).mean()) * 100

    results = {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape
    }
    return results

arima_metrics = evaluate_forecast(test[target_col], arima_forecast, "ARIMA")
sarima_metrics = evaluate_forecast(test[target_col], sarima_forecast, "SARIMA")

results_df = pd.DataFrame([arima_metrics, sarima_metrics])
results_df


## 9. Plot actual vs predicted values


In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(test.index, test[target_col], label="Actual")
plt.plot(arima_forecast.index, arima_forecast, label="ARIMA Forecast")
plt.plot(sarima_forecast.index, sarima_forecast, label="SARIMA Forecast")
plt.title("Actual vs Forecasted Energy Demand")
plt.xlabel("Time")
plt.ylabel(target_col)
plt.legend()
plt.show()


## 10. Inspect a few forecast rows


In [ ]:
comparison_df = pd.DataFrame({
    "Actual": test[target_col],
    "ARIMA Forecast": arima_forecast,
    "SARIMA Forecast": sarima_forecast
})

comparison_df.head(20)


## Notes and suggestions

- If SARIMA performs better than ARIMA, your data likely has a strong seasonal pattern.
- You can tune the orders further using ACF/PACF plots or grid search.
- For a fair comparison with LSTM, keep the **same cleaned dataset**, **same target**, and **same chronological test split**.
